# 04 — MDC case study (Q3): naive vs protocol
Plain inline matplotlib. No styled viz (F6/F7 are Day-9 work).

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from ramanuq.metrics import load_calibrations
from ramanuq.mdc import mdc, to_delta_nd, estimate_sigma_single, estimate_bias

RESULTS = os.path.join('..', 'data', 'synthetic', 'results', 'tierB_grid_results.parquet')
CAL_PATH = os.path.join('..', 'data', 'calibrations', 'calibrations.yaml')
TIERB_DIR = os.path.join('..', 'data', 'synthetic', 'tierB')

df = pd.read_parquet(RESULTS)
cals = load_calibrations(CAL_PATH)

# Excitation wavelength is a fixed property of the suite; read it from truth.
_wls = set()
for _f in os.listdir(TIERB_DIR):
    if _f.endswith('_truth.json'):
        with open(os.path.join(TIERB_DIR, _f)) as _fh:
            _wls.add(json.load(_fh)['wavelength_nm'])
assert len(_wls) == 1, _wls
WAVELENGTH_NM = float(next(iter(_wls)))
MATERIAL_CLASS = 'synthetic_disordered_carbon'
SNR_REGIMES = [15, 50, 200]
CONFIG_COLUMNS = ['baseline', 'lineshape', 'bwf_g', 'peak_set', 'intensity']
WAVELENGTH_NM

In [ ]:
# NAIVE pipeline: linear baseline, single Lorentzian (no BWF), DG peaks, height ratio.
NAIVE_CONFIG = {
    'baseline': 'linear',
    'lineshape': 'lorentzian',
    'bwf_g': False,
    'peak_set': 'DG',
    'intensity': 'height',
}


def select_protocol_config(study_df, snr):
    """PROTOCOL config = DG/area config with the smallest signed-error sd in regime."""
    sub = study_df[
        (study_df.material_class == MATERIAL_CLASS)
        & (study_df.snr_label == snr)
        & (study_df.peak_set == 'DG')
        & (study_df.intensity == 'area')
    ]
    candidates = []
    for keys, grp in sub.groupby(CONFIG_COLUMNS):
        err = grp['error'].to_numpy(dtype=float)
        err = err[np.isfinite(err)]
        if err.size < 2:
            continue
        candidates.append((float(np.std(err, ddof=1)), dict(zip(CONFIG_COLUMNS, keys))))
    candidates.sort(key=lambda t: t[0])
    best_sd, best_cfg = candidates[0]
    return best_cfg, best_sd, candidates

In [ ]:
def _config_str(cfg):
    return '|'.join(f"{k}={cfg[k]}" for k in CONFIG_COLUMNS)


rows = []
for snr in SNR_REGIMES:
    regime = {'material_class': MATERIAL_CLASS, 'snr_label': snr}
    proto_cfg, proto_sd, candidates = select_protocol_config(df, snr)
    print(f'SNR {snr}: selected PROTOCOL = {_config_str(proto_cfg)} '
          f'(smallest DG/area error sd = {proto_sd:.5f} of {len(candidates)} candidates)')

    n_sigma = estimate_sigma_single(df, NAIVE_CONFIG, regime)
    n_bias = estimate_bias(df, NAIVE_CONFIG, regime)
    n_mdc = mdc(n_sigma)
    n_dnd = to_delta_nd(n_mdc, cals, WAVELENGTH_NM)

    p_sigma = estimate_sigma_single(df, proto_cfg, regime)
    p_bias = estimate_bias(df, proto_cfg, regime)
    p_mdc = mdc(p_sigma)
    p_dnd = to_delta_nd(p_mdc, cals, WAVELENGTH_NM)

    rows.append({
        'regime': f'SNR{snr}',
        'naive_config': _config_str(NAIVE_CONFIG),
        'naive_sigma': n_sigma,
        'naive_bias': n_bias,
        'naive_mdc_idig': n_mdc,
        'naive_delta_nd_central': n_dnd[0],
        'protocol_config': _config_str(proto_cfg),
        'protocol_sigma': p_sigma,
        'protocol_bias': p_bias,
        'protocol_mdc_idig': p_mdc,
        'protocol_delta_nd_central': p_dnd[0],
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
cols = [
    'regime',
    'naive_config', 'naive_sigma', 'naive_mdc_idig', 'naive_delta_nd_central',
    'protocol_config', 'protocol_sigma', 'protocol_mdc_idig', 'protocol_delta_nd_central',
]
print(summary[cols].to_string(index=False))

In [ ]:
x = SNR_REGIMES

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, summary['naive_mdc_idig'], marker='o', label='naive')
ax.plot(x, summary['protocol_mdc_idig'], marker='s', label='protocol')
ax.set_xscale('log')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in x])
ax.set_xlabel('SNR')
ax.set_ylabel('MDC (I_D/I_G units)')
ax.set_title('MDC vs SNR — I_D/I_G currency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, summary['naive_delta_nd_central'], marker='o', label='naive')
ax.plot(x, summary['protocol_delta_nd_central'], marker='s', label='protocol')
ax.set_xscale('log')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in x])
ax.set_xlabel('SNR')
ax.set_ylabel('MDC (Delta n_D, cm^-2)')
ax.set_title('MDC vs SNR — Delta n_D currency')
ax.legend()
plt.tight_layout()
plt.show()

## Gate V5 — published-spectrum reproduction (Day 10)

Run the digitized Cançado et al. 2011 (Nano Lett. **11**, 3190; DOI 10.1021/nl201432g)
L_D = 7 nm graphene spectrum through the pipeline in **HEIGHT** mode (the paper defines
I_D/I_G as the peak-height ratio) and compare the extracted I_D/I_G to the published
target (read from `data/digitized/provenance.yaml`) within the pre-registered ±10%
window. The result is written back into `docs/report_data.json` (`gates.V5`), replacing
the `pending_day10` placeholder. The ±10% tolerance is fixed by `validation_plan.md`
and is not changed here.

In [1]:
# === Gate V5 — published-spectrum reproduction ===
# The paper defines I_D/I_G as the peak-HEIGHT ratio, so V5 runs in HEIGHT mode.
# Demonstration config fixed a priori from the paper's STATED method, NOT tuned
# to pass: intensity=height and peak_set=DG are stated by the paper; lineshape=
# lorentzian is the textbook first-order graphene D/G lineshape; baseline=linear
# is the simplest documented choice (the paper specifies no baseline recipe).
import yaml
from ramanuq.io import load_spectrum
from ramanuq.fit import PipelineConfig, fit_spectrum
from ramanuq.metrics import compute_metrics

DIGITIZED_DIR = os.path.join('..', 'data', 'digitized')
REPORT_DATA = os.path.join('..', 'docs', 'report_data.json')

with open(os.path.join(DIGITIZED_DIR, 'provenance.yaml')) as _fh:
    _prov = {s['id']: s for s in (yaml.safe_load(_fh) or {}).get('spectra', [])}
_v5 = _prov['cancado2011_v5']
V5_TARGET = float(_v5['published_id_ig'])        # 1.6, read from provenance
V5_EXCITATION_NM = float(_v5['excitation_nm'])   # 514.5, read from provenance
V5_CSV = os.path.join(DIGITIZED_DIR, 'cancado2011_v5_digitization_a.csv')

V5_TOL_FRAC = 0.10                               # pre-registered ±10% (validation_plan V5)
V5_WINDOW = [V5_TARGET * (1 - V5_TOL_FRAC), V5_TARGET * (1 + V5_TOL_FRAC)]

V5_CONFIG = {'baseline': 'linear', 'lineshape': 'lorentzian',
             'bwf_g': False, 'peak_set': 'DG', 'intensity': 'height'}

_d = pd.read_csv(V5_CSV)
_spec = load_spectrum(_d.iloc[:, 0].to_numpy(), _d.iloc[:, 1].to_numpy(),
                      V5_EXCITATION_NM)
_cfg = PipelineConfig(peak_set='DG', lineshape='lorentzian', bwf_g=False,
                      baseline_method='linear')
_fit = fit_spectrum(_spec, _cfg, n_boot=0, seed=0)
V5_MEASURED = float(compute_metrics(_fit, cals, 'height').id_ig)

V5_PASS = bool(V5_WINDOW[0] <= V5_MEASURED <= V5_WINDOW[1])
V5_RESULT = 'PASS' if V5_PASS else 'MISS'

print(f'Gate V5 — Cançado 2011 L_D=7 nm, HEIGHT mode ({_config_str(V5_CONFIG)})')
print(f'  measured I_D/I_G = {V5_MEASURED:.4f}')
print(f'  published target = {V5_TARGET:.2f}  '
      f'(±{V5_TOL_FRAC:.0%} window [{V5_WINDOW[0]:.3f}, {V5_WINDOW[1]:.3f}])')
print(f'  RESULT: {V5_RESULT}')

# Replace the explicitly-pending gates.V5 placeholder in report_data.json. This
# is the ONLY change to report_data.json (filling a Day-10 placeholder); no
# recomputed value is altered.
with open(REPORT_DATA) as _fh:
    _report = json.load(_fh)
assert _report['gates']['V5']['status'] == 'pending_day10', \
    'gates.V5 is not the expected pending placeholder'
_report['gates']['V5'] = {
    'name': 'Published-spectrum reproduction',
    'status': V5_RESULT,
    'source': 'validation_plan.md Gate V5',
    'tolerance': 'within +/-10% of >=1 digitized published spectrum',
    'spectrum': 'cancado2011_v5',
    'citation_doi': '10.1021/nl201432g',
    'excitation_nm': V5_EXCITATION_NM,
    'config': _config_str(V5_CONFIG),
    'intensity_mode': 'height',
    'measured_idig': V5_MEASURED,
    'target': V5_TARGET,
    'tolerance_frac': V5_TOL_FRAC,
    'window': V5_WINDOW,
    'result': V5_RESULT,
    'note': 'Cancado 2011 L_D=7 nm graphene; paper defines I_D/I_G as the '
            'peak-height ratio. Demonstration config fixed a priori from the '
            'paper stated method (height, DG) + textbook Lorentzian + linear '
            'baseline; not tuned to pass. Asserted on Day 10 per validation_plan.md.',
}
with open(REPORT_DATA, 'w') as _fh:
    json.dump(_report, _fh, indent=2, sort_keys=True)
    _fh.write('\n')
print(f'  wrote gates.V5 -> {REPORT_DATA}')

Gate V5 — Cançado 2011 L_D=7 nm, HEIGHT mode (baseline=linear|lineshape=lorentzian|bwf_g=False|peak_set=DG|intensity=height)
  measured I_D/I_G = 1.5227
  published target = 1.60  (±10% window [1.440, 1.760])
  RESULT: PASS
  wrote gates.V5 -> ../docs/report_data.json
